## Exercise 0 - Gemini and Pydantic for data validation

#### a) Read all the jobs ads into python
Use Python's os and file handling: 
- use the os module or the glob module to find all .txt files in the data directory
- with open(...) loop to read their contents.

In [17]:
# easy approach:  read files and return strings
from pathlib import Path
from typing import Dict

def read_text_files_in_dir(
        dirpath: str, 
        pattern: str = "*.txt", 
        encoding: str = "utf-8") -> Dict[str, str]:
    p = Path(dirpath)
    if not p.exists() or not p.is_dir():
        raise FileNotFoundError(p)
    return {f.name: f.read_text(encoding=encoding) for f in p.glob(pattern) if f.is_file()}

files_dict = read_text_files_in_dir("./data/")
files_dict["ads1.txt"]

"About the team\n\nThe Data Platform team is newly formed and it will have two main roles in play, Data Engineer and Analytics Engineer. We have already developed a comprehensive way of working for ourselves and our stakeholders and built our solutions on a modern data stack using FiveTran, Python, DBT, GCP and Snowflake for data ingestion, storage and modeling with Tableau as our primary data visualization product.\n\n\nWhat you will be doing\n\nWithin the Data Platform team, you will get the chance to shape how Instabee builds its data platform which is serving many stakeholders in the organization and enables us to make better decisions. The data platform is a key asset that we use for multiple data processing purposes; while traditional BI is one use case we deploy, we also service data science investigations and create endpoints for our technology teams to use in our products and services.\n\nAs a Data Engineer you will take the lead on setting integration patterns from sources, a

In [18]:
files_dict

{'ads1.txt': "About the team\n\nThe Data Platform team is newly formed and it will have two main roles in play, Data Engineer and Analytics Engineer. We have already developed a comprehensive way of working for ourselves and our stakeholders and built our solutions on a modern data stack using FiveTran, Python, DBT, GCP and Snowflake for data ingestion, storage and modeling with Tableau as our primary data visualization product.\n\n\nWhat you will be doing\n\nWithin the Data Platform team, you will get the chance to shape how Instabee builds its data platform which is serving many stakeholders in the organization and enables us to make better decisions. The data platform is a key asset that we use for multiple data processing purposes; while traditional BI is one use case we deploy, we also service data science investigations and create endpoints for our technology teams to use in our products and services.\n\nAs a Data Engineer you will take the lead on setting integration patterns fr

#### b) Create a function that uses gemini to summarize a job ad. This function should take in an ad as its parameter and return a summary.

- take the raw text of a job ad
- send it to the Gemini API with a specific prompt
- receive a concise summary back

In [ ]:
from google import genai
from google.genai import types # For configuration options

def summarize_job_ad(ad_text: str) -> str:
    """Uses the Gemini model to generate a concise summary of a job ad."""
    try:
        # **Crucial:** Ensure GEMINI_API_KEY is set in your environment
        # If not set, uncomment and replace "YOUR_API_KEY" below 
        # client = genai.Client(api_key="YOUR_API_KEY") 
        client = genai.Client()
        
        # Craft a clear, direct prompt for the LLM
        prompt = (
            "Analyze the following job advertisement from arbetsförmedlingen.se. "
            "Provide a concise summary of the role, key responsibilities, and mandatory qualifications "
            "in bullet points. Do not include any introductory or concluding sentences."
            "\n\nJOB AD TEXT:\n"
            f"{ad_text}"
        )

        # Call the Gemini API
        response = client.models.generate_content(
            model='gemini-2.5-flash', # A fast and efficient model for text tasks
            contents=prompt,
            config=types.GenerateContentConfig(
                # Set temperature low for factual, less creative output
                temperature=0.1 
            )
        )
        
        return response.text
    
    except Exception as e:
        # Handle cases where the API call fails (e.g., no internet, bad key)
        print(f"An error occurred during Gemini summarization: {e}")
        return "ERROR: Could not generate summary."


In [11]:
# Example of testing the function
summary = summarize_job_ad(files_dict["ads1.txt"])
print(summary)

*   **Role Summary:**
    *   Data Engineer with a strong interest and experience in Analytics Engineering, responsible for shaping and building Instabee's data platform to serve various stakeholders and enable data-driven decision-making.

*   **Key Responsibilities:**
    *   Lead the setting of integration patterns and build/migrate source integrations for the data platform.
    *   Set up and maintain data platform infrastructure.
    *   Drive initiatives to introduce data product principles such as discoverability and trustworthiness.
    *   Design and build data models that support business processes, ensuring data governance and trust.
    *   Work with data transformation and modeling techniques using DBT with SQL and Python.
    *   Contribute to analytical projects to help answer hypotheses or solve stakeholder ideas.

*   **Mandatory Qualifications:**
    *   Experience in both Data Engineering and Analytics Engineering.
    *   Proven experience building modern data platf

#### c) Create and export markdown files for each job ad and its corresponding summary.
- iterate through all the loaded job ads
- generate a summary for each one using your function
- write the original ad and its summary into a new Markdown (.md) file

In [1]:
import os

def create_and_export_markdown(job_ads_dict: dict):
    """Generates summaries and exports them with the original text to Markdown files."""
    output_folder = "summaries"
    # Create the output directory if it doesn't exist
    os.makedirs(output_folder, exist_ok=True) 

    for filename, ad_text in job_ads_dict.items():
        # Use the filename to create a clean title for the Markdown file
        title = filename.replace('.txt', '').replace('_', ' ').title() 
        output_filename = filename.replace('.txt', '.md')
        
        # **1. Generate the Summary**
        print(f"Generating summary for {filename}...")
        summary = summarize_job_ad(ad_text)
        
        # **2. Format the Content as Markdown**
        markdown_content = f"""# 📝 Job Ad Analysis: {title}

        ---

        ## **Summary by Gemini**

        {summary}

        ---

        ## **Original Job Advertisement**

        {ad_text}

        """
        # **3. Save the Markdown File**
        output_path = os.path.join(output_folder, output_filename)
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(markdown_content)
                
        print(f"Successfully exported: {output_path}")


In [16]:
# 2. Process and export
if files_dict:
    create_and_export_markdown(files_dict)
else:
    print("No job ads were loaded. Please check your 'data' folder.")

Generating summary for ads1.txt...
Successfully exported: summaries\ads1.md
Generating summary for ads2.txt...
Successfully exported: summaries\ads2.md
Generating summary for ads3.txt...
Successfully exported: summaries\ads3.md


####- d) Try to take other job ads from arbetsförmedlingen.se and see how well your function performs.

In [22]:
# 2. Process and export
if files_dict:
    create_and_export_markdown(files_dict)
else:
    print("No job ads were loaded. Please check your 'data' folder.")

Generating summary for ads1.txt...
Successfully exported: summaries\ads1.md
Generating summary for ads2.txt...
Successfully exported: summaries\ads2.md
Generating summary for ads3.txt...
Successfully exported: summaries\ads3.md
Generating summary for ads4.txt...
Successfully exported: summaries\ads4.md
